# Document Review Assistant with BERT

This notebook demonstrates the same pipeline used by the Streamlit application. It uses three BERT roles:

1. BERT fine-tuned for Named Entity Recognition.
2. BERT fine-tuned for extractive question answering.
3. Base BERT embeddings for ambiguity assessment, extractive summary, and clause review.

The model either identifies labels, extracts an answer from source text, or compares embeddings.

## Pipeline architecture

`Document -> text extraction -> chunking -> BERT task models -> evidence shown to the user`

Long documents are split into overlapping chunks because BERT has a limited input context. The final application accepts TXT, PDF, and DOCX files.

In [1]:
from dataclasses import asdict
from pathlib import Path

from pipeline import BertDocumentAssistant, extract_text, normalise_text, split_for_qa

# The first run downloads the three BERT model checkpoints.
assistant = BertDocumentAssistant()

## 1. Load a document

Keep `USE_FILE` as `False` to use the small demonstration text. After selecting the two final source documents, set it to `True` and update `FILE_PATH`.

In [2]:
USE_FILE = False
FILE_PATH = Path('sample_documents/Sample Contract.docx')

demo_text = '''
Northstar Solutions will provide monthly software support to Brightway Retail.
Priya Mehta, Operations Manager at Brightway Retail, may be contacted at priya.mehta@brightway.example or +91 98765 43210.
The company will pay invoices within 30 calendar days of receipt.
Either party may terminate this agreement by giving 14 days written notice.
The supplier should respond promptly when practical and take reasonable action as needed.
All confidential customer information must be protected from unauthorised disclosure.
'''

# if USE_FILE:
#     document_text = normalise_text(extract_text(FILE_PATH.name, FILE_PATH.read_bytes()))
# else:
#     document_text = normalise_text(demo_text)

document_text = normalise_text(extract_text(FILE_PATH.name, FILE_PATH.read_bytes()))

print(document_text)
print(f'QA chunks: {len(split_for_qa(document_text))}')

SERVICE CONTRACT
Contract No. 567890
THIS SERVICE CONTRACT entered into as of 21-07-26 by and between MERCY CORPS, a State of Washington, U.S.A. nonprofit corporation having its principal office in Portland, Oregon, U.S.A. (“Mercy Corps”) and Rudra Company (“Contractor”) is as follows:
Defined Terms. Each of the following terms has the meaning given to such term on Schedule I attached hereto: Authorized Representative, Payment Terms, Services and SOW. “Contract” means this Service Contract as amended, modified or supplemented from time to time taken together with its Schedules. Additional terms may be defined throughout this Contract.
Delivery of Services.
Contractor will perform the Services, and Mercy Corps will pay for the Services, in accordance with the terms and conditions and within the Performance Period set forth in this Contract and the Statement of Services.
Contractor will perform all Services through the services of Contractor’s employees. Contractor will not delegate or s

## 2. Named Entity Recognition and structured values

The NER model finds contextual entities such as people, organisations, and locations. Transparent format matching supplements it for values such as emails and phone numbers, which have predictable structures.

In [3]:
findings = assistant.scan_document(document_text)
[asdict(finding) for finding in findings]

[{'text': '21-07-26', 'category': 'Date', 'confidence': None},
 {'text': 'Dhurandhar',
  'category': 'Location',
  'confidence': 0.769002377986908},
 {'text': 'Oregon', 'category': 'Location', 'confidence': 0.9992372989654541},
 {'text': 'Portland',
  'category': 'Location',
  'confidence': 0.9981114864349365},
 {'text': 'State', 'category': 'Location', 'confidence': 0.705877423286438},
 {'text': 'State of Oregon',
  'category': 'Location',
  'confidence': 0.7284074425697327},
 {'text': 'U. S', 'category': 'Location', 'confidence': 0.8836708068847656},
 {'text': 'U. S.', 'category': 'Location', 'confidence': 0.9947749376296997},
 {'text': 'United States',
  'category': 'Location',
  'confidence': 0.9993937015533447},
 {'text': 'and Records',
  'category': 'Organisation',
  'confidence': 0.9643179178237915},
 {'text': 'Contractor',
  'category': 'Organisation',
  'confidence': 0.9372488856315613},
 {'text': 'Intellectual Property Rights',
  'category': 'Organisation',
  'confidence': 0.

In [4]:
# A redacted preview makes the scan easier to demonstrate.
print(assistant.redact_text(document_text, findings))

SERVICE CONTRACT
Contract No. 567890
THIS SERVICE CONTRACT entered into as of [DATE] by and between [ORGANISATION], a [LOCATION] [ORGANISATION], U.S.A. nonprofit corporation having its principal office in [LOCATION], [LOCATION], U.S.A. (“[ORGANISATION]”) and [ORGANISATION] (“[ORGANISATION]”) is as follows:
Defined Terms. Each of the following terms has the meaning given to such term on Schedule I attached hereto: Authorized Representative, Payment Terms, [ORGANISATION] and SOW. “Contract” means this Service Contract as amended, modified or supplemented from time to time taken together with its Schedules. Additional terms may be defined throughout this Contract.
Delivery of [ORGANISATION].
[ORGANISATION] will perform the [ORGANISATION], and [ORGANISATION] will pay for the [ORGANISATION], in accordance with the terms and conditions and within the Performance Period set forth in this Contract and the [ORGANISATION].
[ORGANISATION] will perform all [ORGANISATION] through the [ORGANISATION]

## 3. Ask a plain-English question

The QA model is extractive. It selects an answer span from the document rather than composing a new answer.

In [5]:
question = 'How long does Mercy Corps have to pay an invoice?'
# What happens when either party breaches the contract?
# What is the Contract number?
# What is the name of Mercy Corps’ Chief Contract Officer?
answer = assistant.answer_question(question, document_text)
answer

{'answer': '30 days',
 'confidence': 0.9941668510437012,
 'context': 'the invoice. Such notice shall clearly state the specific Services disputed, and Mercy Corps’ reason for disputing the performance of the Services. If both parties accept the dispute of the invoice, they shall agree in writing as to the steps required of Contractor to ensure that the performance of the disputed Services is subsequently completed in accordance with the Additional Terms, and the time required of Contractor to complete the Services. Except as otherwise provided in the Statement of Services, Mercy Corps will pay each invoice (or adjusted invoice if the subject of dispute) in accordance with the Payment Terms within 30 days after the later of (i) receipt of the invoice or (ii) resolution of the items set forth in the notice of disputed charges. Mercy Corps may off-set any amount it owes Contractor against any amount Contractor owes Mercy Corps. Taxes, Duties and Expenses. Except as otherwise provided in t

## 4. Writing assistant

Base BERT embeddings compare a proposed sentence with reference examples of clear and potentially ambiguous business wording. It highlights vague terms for human review and does not rewrite the sentence.

In [6]:
sentence = 'The supplier should respond promptly when practical.'
assistant.assess_ambiguity(sentence)

{'verdict': 'Potentially ambiguous',
 'clear_score': 0.6527790427207947,
 'ambiguous_score': 0.7915948033332825,
 'cues': ['promptly', 'practical']}

## 5. Additional features

The summary ranks sentences already present in the source document. Clause review compares document chunks with five clause descriptions using the same base BERT embedding model. Both results are review aids rather than legal conclusions.

In [7]:
summary = assistant.extractive_summary(document_text, max_sentences=15)
for index, source_sentence in enumerate(summary, start=1):
    print(f'{index}. {source_sentence}')

1. Contractor will perform the Services, and Mercy Corps will pay for the Services, in accordance with the terms and conditions and within the Performance Period set forth in this Contract and the Statement of Services.
2. Contractor will not delegate or subcontract any Services to be provided to Mercy Corps without Mercy Corps’ prior written consent.
3. Contractor will not change the Key Personnel without prior notice and an amendment to this Contract specifying the change.
4. No deviation, substitution or change is permitted without Mercy Corps’ prior written consent; provided that Mercy Corps may terminate, suspend, increase or decrease the scope of Contractor's performance under the SOW by written notice to Contractor specifying the changes.
5. If any change causes an increase or decrease in the cost of, or the time required for, Contractor's performance, an equitable adjustment may be made in the SOW or Payment Terms or both, if such adjustment is set forth in an amendment signed 

In [8]:
clause_review = assistant.review_clauses(document_text)
for item in clause_review:
    print(f"{item['category']}: similarity={item['similarity']:.3f}")
    print(item['evidence'])
    print()

Payment: similarity=0.727
forth in the Statement of Services (Schedule I). Final invoices must be submitted within 60 days of the end date of the Contract. Contractor recognizes that in many cases Mercy Corps’ donor will not reimburse Mercy Corps for invoices submitted beyond 60 days after the termination of a contract and therefore Mercy Corps will have no obligation to pay any portion of invoices received more than 60 days after the end date of the Contract. Each invoice will include (i) the Contract Number; (ii) Contractor’s name and address; (iii) a

Termination: similarity=0.725
any Services to be provided to Mercy Corps without Mercy Corps’ prior written consent. Contractor agrees that including the specific individuals named (if any) as Key Personnel in Schedule I is a material part of the bargain. Contractor will not change the Key Personnel without prior notice and an amendment to this Contract specifying the change. Mercy Corps may withhold its consent to substitute personnel

## Limitations and responsible use

- NER predictions and similarity scores can be wrong, so a human must review results.
- The QA confidence score is not a factual guarantee.
- The application does not give legal, HR, medical, or compliance advice.